# API And Vision Runtime Check

This notebook does two things:

1. Load `base_url / api_key / model` from the project config you were using.
2. Run a small smoke test for the vision tools that failed in model3: `MSCN`, `SM3Det`, `RemoteSAM`, `ChangeOS`.

Expected usage:

- Open this notebook from the repository root.
- Prefer a kernel bound to `E:/miniconda3/envs/earth-bench-skill-eval/python.exe`.
- Run cells from top to bottom.


In [8]:
from pathlib import Path
import csv
import importlib
import importlib.util
import json
import os
import sys
import time
import traceback
from pprint import pprint

ROOT = Path.cwd()
PROJECT_CODE_ROOT = ROOT / "project_skills"
VERIFY_CONFIG = PROJECT_CODE_ROOT / "_history_cleanup_20260320" / "runs" / "_verify_q1_gpt54_20260320_140018" / "system.local.verify.json"
FALLBACK_CONFIG = PROJECT_CODE_ROOT / "configs" / "system.json"
CONFIG_PATH = VERIFY_CONFIG if VERIFY_CONFIG.exists() else FALLBACK_CONFIG
TEMP_DIR = ROOT / "_notebook_runtime_check"
TEMP_DIR.mkdir(parents=True, exist_ok=True)

print("ROOT:", ROOT)
print("PROJECT_CODE_ROOT:", PROJECT_CODE_ROOT)
print("CONFIG_PATH:", CONFIG_PATH)
print("TEMP_DIR:", TEMP_DIR)
print("PYTHON:", sys.executable)

if "earth-bench-skill-eval" not in sys.executable.replace("\\", "/"):
    print("WARNING: current kernel is not the expected earth-bench-skill-eval environment.")


ROOT: d:\skills-evo\project_skills
PROJECT_CODE_ROOT: d:\skills-evo\project_skills\project_skills
CONFIG_PATH: d:\skills-evo\project_skills\project_skills\_history_cleanup_20260320\runs\_verify_q1_gpt54_20260320_140018\system.local.verify.json
TEMP_DIR: d:\skills-evo\project_skills\_notebook_runtime_check
PYTHON: e:\miniconda3\envs\earth-bench-skill-eval\python.exe


In [9]:
API_KEY_OVERRIDE = None

with CONFIG_PATH.open("r", encoding="utf-8-sig") as f:
    cfg = json.load(f)

actor_cfg = cfg["actor"]
BASE_URL = actor_cfg["base_url"]
MODEL = actor_cfg["model"]
API_KEY = API_KEY_OVERRIDE or os.environ.get("SKILL_EVAL_API_KEY") or actor_cfg["api_key"]
TIMEOUT_SECONDS = int(actor_cfg.get("timeout_seconds", 180))

def mask_key(value: str) -> str:
    if not value:
        return "<empty>"
    if len(value) <= 10:
        return value[0:2] + "***"
    return value[:6] + "..." + value[-4:]

print({
    "base_url": BASE_URL,
    "model": MODEL,
    "api_key": mask_key(API_KEY),
    "timeout_seconds": TIMEOUT_SECONDS,
})


{'base_url': 'http://35.220.164.252:3888/v1', 'model': 'gpt-5.4', 'api_key': 'sk-Jhr...l9lQ', 'timeout_seconds': 180}


In [10]:
package_status = {}
for name in ["openai", "pandas", "fastmcp", "numpy", "rasterio"]:
    try:
        importlib.import_module(name)
        package_status[name] = "OK"
    except Exception as exc:
        package_status[name] = f"{type(exc).__name__}: {exc}"

pprint(package_status)


{'fastmcp': 'OK',
 'numpy': 'OK',
 'openai': 'OK',
 'pandas': "ModuleNotFoundError: No module named 'pandas'",
 'rasterio': 'OK'}


In [11]:
llm_result = {}
try:
    from openai import OpenAI
    client = OpenAI(base_url=BASE_URL, api_key=API_KEY, timeout=TIMEOUT_SECONDS)
    started = time.time()
    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": "Reply with exactly OK."}],
        temperature=0,
        max_tokens=8,
    )
    content = response.choices[0].message.content if response.choices else "<no choices>"
    llm_result = {
        "ok": True,
        "seconds": round(time.time() - started, 3),
        "content": content,
    }
except Exception as exc:
    llm_result = {
        "ok": False,
        "error": f"{type(exc).__name__}: {exc}",
        "traceback": traceback.format_exc(),
    }

pprint(llm_result)


{'error': "PermissionDeniedError: Error code: 403 - {'error': {'message': "
          "'用户额度不足, 剩余额度: ＄-1.250358 (request id: "
          "20260323002351617208961pfUoMm54)', 'type': 'new_api_error', "
          "'param': '', 'code': 'insufficient_user_quota'}}",
 'ok': False,
 'traceback': 'Traceback (most recent call last):\n'
              '  File '
              '"C:\\Users\\HP\\AppData\\Local\\Temp\\ipykernel_3860\\3382413753.py", '
              'line 6, in <module>\n'
              '    response = client.chat.completions.create(\n'
              '               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^\n'
              '  File '
              '"e:\\miniconda3\\envs\\earth-bench-skill-eval\\Lib\\site-packages\\openai\\_utils\\_utils.py", '
              'line 286, in wrapper\n'
              '    return func(*args, **kwargs)\n'
              '           ^^^^^^^^^^^^^^^^^^^^^\n'
              '  File '
              '"e:\\miniconda3\\envs\\earth-bench-skill-eval\\Lib\\site-packages\\openai\\

In [12]:
def load_perception_module(temp_dir: Path):
    tools_dir = PROJECT_CODE_ROOT / "agent" / "tools"
    module_path = tools_dir / "Perception.py"
    if str(tools_dir) not in sys.path:
        sys.path.insert(0, str(tools_dir))
    old_argv = sys.argv[:]
    try:
        sys.argv = [str(module_path), "--temp_dir", str(temp_dir)]
        spec = importlib.util.spec_from_file_location("notebook_perception", module_path)
        module = importlib.util.module_from_spec(spec)
        assert spec is not None and spec.loader is not None
        spec.loader.exec_module(module)
        return module
    finally:
        sys.argv = old_argv

perception = load_perception_module(TEMP_DIR)
print("Loaded perception module from:", perception.__file__)


Loaded perception module from: d:\skills-evo\project_skills\project_skills\agent\tools\Perception.py


In [13]:
SAMPLES = {
    "MSCN": PROJECT_CODE_ROOT / "benchmark" / "data" / "question189" / "J.jpg",
    "SM3Det": PROJECT_CODE_ROOT / "benchmark" / "data" / "question214" / "A.png",
    "RemoteSAM": PROJECT_CODE_ROOT / "benchmark" / "data" / "question226" / "478549_4934011_2048_32610_sport_soccer.jpg",
    "ChangeOS_pre": PROJECT_CODE_ROOT / "benchmark" / "data" / "question219" / "t1.png",
    "ChangeOS_post": PROJECT_CODE_ROOT / "benchmark" / "data" / "question219" / "t2.png",
}

for name, path in SAMPLES.items():
    print(name, path.exists(), path)

def brief(value, limit=500):
    text = repr(value)
    return text if len(text) <= limit else text[:limit] + " ..."

def run_check(name, fn):
    started = time.time()
    try:
        value = fn()
        return {
            "ok": True,
            "seconds": round(time.time() - started, 3),
            "result": brief(value),
        }
    except Exception as exc:
        return {
            "ok": False,
            "seconds": round(time.time() - started, 3),
            "error": f"{type(exc).__name__}: {exc}",
            "traceback": traceback.format_exc(limit=4),
        }

tool_results = {
    "MSCN": run_check("MSCN", lambda: perception.MSCN(str(SAMPLES["MSCN"]))),
    "SM3Det": run_check("SM3Det", lambda: perception.SM3Det(str(SAMPLES["SM3Det"]), "tennis court")),
    "RemoteSAM": run_check("RemoteSAM", lambda: perception.RemoteSAM(str(SAMPLES["RemoteSAM"]), "the football field located on the westernmost side")),
    "ChangeOS": run_check("ChangeOS", lambda: perception.ChangeOS(str(SAMPLES["ChangeOS_pre"]), str(SAMPLES["ChangeOS_post"]), "notebook_smoke/change_mask.png")),
}

pprint(tool_results)


MSCN True d:\skills-evo\project_skills\project_skills\benchmark\data\question189\J.jpg
SM3Det True d:\skills-evo\project_skills\project_skills\benchmark\data\question214\A.png
RemoteSAM True d:\skills-evo\project_skills\project_skills\benchmark\data\question226\478549_4934011_2048_32610_sport_soccer.jpg
ChangeOS_pre True d:\skills-evo\project_skills\project_skills\benchmark\data\question219\t1.png
ChangeOS_post True d:\skills-evo\project_skills\project_skills\benchmark\data\question219\t2.png
{'ChangeOS': {'error': "ModuleNotFoundError: No module named 'pandas'",
              'ok': False,
              'seconds': 0.0,
              'traceback': 'Traceback (most recent call last):\n'
                           '  File '
                           '"C:\\Users\\HP\\AppData\\Local\\Temp\\ipykernel_3860\\1211614404.py", '
                           'line 19, in run_check\n'
                           '    value = fn()\n'
                           '            ^^^^\n'
                     

In [14]:
def contains_pandas_failure(item: dict) -> bool:
    blob = "\n".join(str(v) for v in item.values())
    return "No module named 'pandas'" in blob or 'No module named "pandas"' in blob

summary = {
    "llm_ok": llm_result.get("ok", False),
    "pandas_import_ok": package_status.get("pandas") == "OK",
    "tool_failures": {k: v for k, v in tool_results.items() if not v.get("ok")},
    "tools_with_pandas_failure": [k for k, v in tool_results.items() if contains_pandas_failure(v)],
}

pprint(summary)

if summary["llm_ok"]:
    print("LLM endpoint looks reachable with the configured key.")
else:
    print("LLM endpoint/key check failed. See llm_result above.")

if summary["tools_with_pandas_failure"]:
    print("Vision runtime is blocked by missing pandas for:", summary["tools_with_pandas_failure"])
elif summary["tool_failures"]:
    print("Vision runtime still has failures, but not the pandas signature. Inspect tool_results above.")
else:
    print("All four vision tools returned without raising a notebook-level exception.")


{'llm_ok': False,
 'pandas_import_ok': False,
 'tool_failures': {'ChangeOS': {'error': 'ModuleNotFoundError: No module named '
                                         "'pandas'",
                                'ok': False,
                                'seconds': 0.0,
                                'traceback': 'Traceback (most recent call '
                                             'last):\n'
                                             '  File '
                                             '"C:\\Users\\HP\\AppData\\Local\\Temp\\ipykernel_3860\\1211614404.py", '
                                             'line 19, in run_check\n'
                                             '    value = fn()\n'
                                             '            ^^^^\n'
                                             '  File '
                                             '"C:\\Users\\HP\\AppData\\Local\\Temp\\ipykernel_3860\\1211614404.py", '
                                             '